In [ ]:
import os
import shutil
import trainer.trainer
from TTS.tts.configs.glow_tts_config import GlowTTSConfig
from TTS.config.shared_configs import BaseDatasetConfig 
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.glow_tts import GlowTTS
from TTS.utils.audio import AudioProcessor
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from trainer import Trainer, TrainerArgs

def silent_error(func):
    def wrapper(path, *args, **kwargs):
        try: return func(path, *args, **kwargs)
        except Exception: return None
    return wrapper

os.unlink = silent_error(os.unlink)
os.remove = silent_error(os.remove)
os.rmdir = silent_error(os.rmdir)
shutil.rmtree = silent_error(shutil.rmtree)
trainer.trainer.Trainer.init_training = lambda self, *args, **kwargs: (args[2], {})


# 1. Paths
root_path = r"C:\TTS-glowtts\tamil_dataset"
output_path = os.path.join(root_path, "output_tamil")
pretrained_path = r"C:\TTS-glowtts\Base_Model\model_file.pth"
config_path = r"C:\TTS-glowtts\Base_Model\config.json"

# 2. Load Config
config = GlowTTSConfig()
config.load_json(config_path)
ap = AudioProcessor(**config.audio)

# 3. Initialize Tokenizer (This reads the Tamil characters from config)
tokenizer, config = TTSTokenizer.init_from_config(config)

# 4. Load Data
dataset_config = BaseDatasetConfig(
    dataset_name="tamil_dataset",
    path=root_path,
    meta_file_train="metadata_train.csv",
    meta_file_val="metadata_test.csv",
    language="ta", 
    formatter="ljspeech"
)
train_samples, eval_samples = load_tts_samples([dataset_config], eval_split=False)

print(f"DEBUG: Found {len(train_samples)} training samples")

if len(train_samples) == 0:
    raise RuntimeError(" STOP! No training samples were found. Check your CSV filename and folder structure.")

# 5. Initialize Model
model = GlowTTS(config, ap, tokenizer=tokenizer, speaker_manager=None)

# 6. Setup Trainer
config.output_path = output_path
trainer = Trainer(
    TrainerArgs(restore_path=pretrained_path, skip_train_epoch=False),
    config,
    output_path,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)

# 7. Start Training
trainer.fit()

c:\TTS-glowtts\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEBUG: Found 2101 training samples


 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: True
 | > Precision: fp16
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 12
 | > Num. of Torch Threads: 6
 | > Torch seed: 54321
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
 > Start Tensorboard: tensorboard --logdir=C:\TTS-glowtts\tamil_dataset\output_tamil\glow-tts-rel_pos_transformer-March-14-2026_10+59AM-0000000
c:\TTS-glowtts\.venv\lib\site-packages\trainer\trainer.py:552: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
 > Restoring from model_file.pth ...
 > Restoring Model...
 > Partial model initialization...
 | > Layer missing in the model definition: decoder.flows.2.start.weight_g
 | > Layer missing in the model definition: decoder.flows.2.start.weight_v
 | > Layer missing in the model definitio